# 10. AI Chips Overview and Alternatives | 算力现状与替代方案

**难度：** Medium | **环境：** CPU-first | **标签：** `硬件系统`, `AI 芯片`, `异构算力` | **目标人群：** 硬件约束学习者

![加速器候选的可用性判断](../docs/public/01_Hardware_Math_and_Systems/10_accelerator_readiness.svg)

---

## 本节导读

硬件选型不能只比较峰值算力和显存容量。软件栈成熟度、算子覆盖、模型迁移、通信能力和运维条件都会影响候选是否能够执行目标工作负载。

本节建立“设备能力 → 软件与算子支持 → 迁移风险 → 单位工作负载成本”的筛选关系。完成后，你应能先排除不可执行的候选，再把保留方案交给部署与异构系统专题进行真实运行时验证。

**关键词：** 设备生态、软件栈、迁移风险


## 前置阅读
**导语：** 这一页先把硬件规格、软件栈成熟度和迁移成本放到同一个框架里，再判断哪条路线更适合落地。
- [Group 1B: Single-Device Execution, Memory and Data Access | 1B: 单设备执行、内存与访存](./1B.md)
- [Group 1C: Communication, Parallelism and Fault Tolerance | 1C: 通信、并行与容错](./1C.md)
- [Group 1F: Hardware Adaptation and System Constraints | 1F: 硬件适配与系统约束](./1F.md)

## Q1：选芯片时为什么不能只看算力峰值？


纸面峰值只说明“理论上能跑多快”，不说明“真实工作负载里能不能持续跑快”。

实际选型还要看：
- 软件栈是否成熟；
- 编译器和 kernel 是否可用；
- 通信和调度能力是否匹配；
- 迁移和维护成本是否可控。

所以选型不是单一性能数字的比较，而是端到端可落地性的比较。


### Q1 可行性检查：峰值比较之前必须满足哪些条件

先看能不能稳定跑，再看峰值。

In [ ]:
def accelerator_feasibility(model_bytes, device_memory_bytes, required_ops, supported_ops, required_dtype, supported_dtypes):
    """在比较峰值前检查容量、算子和 dtype 三个硬条件。"""
    if model_bytes <= 0 or device_memory_bytes <= 0:
        raise ValueError('memory bytes must be positive')
    reasons = []
    if model_bytes > device_memory_bytes:
        reasons.append('insufficient_memory')
    missing_ops = sorted(set(required_ops) - set(supported_ops))
    if missing_ops:
        reasons.append('missing_ops')
    if required_dtype not in supported_dtypes:
        reasons.append('unsupported_dtype')
    return {'feasible': not reasons, 'reasons': reasons, 'missing_ops': missing_ops}

ready = accelerator_feasibility(8e9, 16e9, {'matmul', 'softmax'}, {'matmul', 'softmax'}, 'bf16', {'fp16', 'bf16'})
blocked = accelerator_feasibility(20e9, 16e9, {'matmul', 'softmax'}, {'matmul'}, 'bf16', {'fp16'})
assert ready['feasible'] is True
assert blocked['feasible'] is False
print('ready:', ready)
print('blocked:', blocked)

## Q2：为什么软件栈成熟度会直接影响硬件可用性？


硬件不是孤立存在的，真正决定效率的是软件栈是否把它用起来。可用性判断的关键，不是某一层名词是否存在，而是 driver / compiler / runtime / kernel 是否能闭环。

如果编译器、驱动、通信库和 kernel 支持不完整，硬件峰值往往到不了生产场景里。相反，软件栈成熟的设备即使纸面指标没那么夸张，也可能在端到端流程里更稳。

所以“可替代方案”判断里，软件生态本身就是性能的一部分。


### Q2 依赖链检查：软件栈缺口发生在哪一层

硬件能力要经过软件栈才能落地。

In [ ]:
def stack_readiness(driver, compiler, runtime, kernel, communication, kernel_coverage=1.0):
    """按依赖顺序返回首个阻塞层，并检查关键算子覆盖率。"""
    layers = {'driver': driver, 'compiler': compiler, 'runtime': runtime, 'kernel': kernel, 'communication': communication}
    if not 0 <= kernel_coverage <= 1:
        raise ValueError('kernel_coverage must be in [0, 1]')
    missing = [name for name, available in layers.items() if not available]
    ready = not missing and kernel_coverage == 1.0
    first_blocker = missing[0] if missing else ('kernel_coverage' if not ready else None)
    return {'ready': ready, 'missing': missing, 'first_blocker': first_blocker, 'kernel_coverage': kernel_coverage}

cases = {
    'stack_ready': {'driver': True, 'compiler': True, 'runtime': True, 'kernel': True, 'communication': True},
    'compiler_gap': {'driver': True, 'compiler': False, 'runtime': True, 'kernel': True, 'communication': True},
    'communication_gap': {'driver': True, 'compiler': True, 'runtime': True, 'kernel': True, 'communication': False},
}

for name, support in cases.items():
    print(name, '->', stack_readiness(**support))
assert stack_readiness(**cases['stack_ready'])['ready'] is True
assert stack_readiness(**cases['compiler_gap'])['first_blocker'] == 'compiler'


## Q3：什么时候替代方案比主流 GPU 更值得考虑？


当预算、供应链、功耗、部署环境或合规要求发生变化时，替代方案就不只是备选，而可能成为主方案。

但替代方案是否值得选，仍然要回到三件事：
- 能不能跑；
- 跑得稳不稳；
- 长期维护成本高不高。

所以这里的核心不是“谁最强”，而是“谁更适合当前约束”。


### Q3 成本计算：可行候选的单位工作负载成本是多少

把硬件能力、软件栈成熟度和迁移成本放在一起，看看什么时候替代方案更值得考虑。


In [ ]:
def cost_per_million_requests(capex, lifetime_hours, power_kw, electricity_per_kwh, engineering_cost, requests_per_second, availability):
    """把设备、能耗和迁移工程成本换算成每百万成功请求成本。"""
    if min(capex, lifetime_hours, power_kw, electricity_per_kwh, engineering_cost, requests_per_second) < 0:
        raise ValueError('cost inputs must be non-negative')
    if lifetime_hours <= 0 or requests_per_second <= 0 or not 0 < availability <= 1:
        raise ValueError('invalid lifetime, throughput or availability')
    hourly_cost = capex / lifetime_hours + power_kw * electricity_per_kwh + engineering_cost / lifetime_hours
    successful_requests_per_hour = requests_per_second * 3600 * availability
    return hourly_cost / successful_requests_per_hour * 1_000_000

candidates = {
    'baseline': cost_per_million_requests(30_000, 15_000, 0.7, 0.8, 0, 20, 0.99),
    'alternative': cost_per_million_requests(18_000, 15_000, 0.5, 0.8, 12_000, 18, 0.97),
}
selected = min(candidates, key=candidates.get)
assert all(cost > 0 for cost in candidates.values())
print('cost per million requests:', candidates)
print('selected:', selected)


## ⚠️ 常见误区

- 不要把纸面峰值当成真实吞吐。
- 软硬件栈成熟度会直接影响落地效果。
- 迁移成本往往比初始采购价更重要。
- 选型本质上是在一组约束下做最优折中。

## 相关阅读
**导语：** 如果想继续把选型、编译和通信侧的判断补完整，可以接着看这些页。
- [05. Triton 性能调优与基准测试 (Autotune & Profiling)](../03_Triton_Kernels/05_Triton_Autotune_and_Profiling.ipynb)
- [15. CUDA Custom Kernel Intro | CUDA 自定义算子入门](../04_CUDA_and_System_Optimization/15_CUDA_Custom_Kernel_Intro.ipynb)
- [09. Distributed Communication Primitives | 分布式进阶：多机通信原语实战 (All-Reduce, All-Gather)](../04_CUDA_and_System_Optimization/19_Distributed_Communication_Primitives.ipynb)